***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](#)
    * 下一节：[2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)

***


# 第 2 章：干涉测量的数学工具箱<a id='math:sec:intro'></a>


第 1 章介绍了射电干涉测量的科学背景和仪器基础；本章建立描述信号、采样过程和反演问题所需的数学框架。核心内容包括复数与相位、傅里叶表示、离散化、矩阵方程、最小二乘估计和统计不确定度。这些内容共同构成后续可见度、成像与校准推导的数学语言。

本章按照干涉测量的信息处理链组织。电磁波的振幅与相位由复数表示；两个阵元的相关把相位差写入复可见度；可见度在空间频率平面上采样天空亮度；有限采样、噪声和离散化进一步使成像与校准成为矩阵形式的反问题。傅里叶变换、卷积、相关、采样理论和最小二乘法分别描述这条链中的一个环节。

本章以微积分和基础线性代数为先修知识，重点讨论第 4 至第 8 章将反复使用的定义、变换规则、适用条件与数值近似。公式推导保留影响物理解释和计算实现的关键步骤；纯数学证明仅在有助于说明适用范围时给出。


#### 数学工具的组织主线

本章按照从信号表示到反演求解的顺序组织。复数统一表示振幅与相位；傅里叶分析联系天空方向结构与基线采样；卷积和相关描述仪器响应与信号比较；离散傅里叶变换和采样理论给出连续关系的离散数值形式；线性代数、最小二乘和统计方法则组织带噪数据、模型参数及求解算法。

![第二章的数学工具链](figures/chapter2_math_toolchain.png)

**图 2.0.1**：第二章的数学工具链。图中箭头表示射电干涉测量中常见的处理顺序，而非严格的逻辑依赖关系：从电场相位出发，经复相关形成可见度，再通过傅里叶分析、采样与反演进入成像和校准。


#### 阅读路径

第二章可根据先修基础和课程目标选择阅读路径。2.12--2.14 为几何与计算专题，2.15 直接承接 2.11，属于带噪反演的核心内容。建议按下表安排：

| 路径 | 页面 | 完成标准 |
|:---|:---|:---|
| 核心 | 2.1，2.4--2.11，2.15，2.y.1--2.y.6 | 能固定符号约定，解释有限采样，并用协方差和条件数判断一个反演解 |
| 基础补充 | 2.2--2.3，2.12--2.13 | 在狄拉克 $\delta$ 函数、窗函数、傅里叶级数、立体角或球面几何基础不足时按需学习 |
| 计算专题 | 2.14，基 2 FFT 作业，2.y 后续示例 | 从定义实现算法，并依据数值残差而非图形相似性判断结果正确性 |

本科高年级课程应完成核心路径；研究生课程或计算方向课程可加入计算专题。基础补充部分用于弥补不同学习者的先修差异，可根据实际需要选读。

#### 全书数学约定<a id='math:sec:canonical_conventions'></a>

下表给出后续章节采用的统一约定。局部推导可以更换变量名称，但必须明确指数符号、共轭次序、复方差和权重含义；比较外部软件或文献公式时，应首先完成符号约定的对应转换。

| 对象 | 本书约定 | 必须同时说明的边界 |
|:---|:---|:---|
| 连续傅里叶变换对 | $F(s)=\int f(x)e^{-2\pi ixs}dx$；$f(x)=\int F(s)e^{+2\pi ixs}ds$ | $x$ 与 $s$ 的单位互为倒数；二维天空与 $uv$ 平面沿用相同符号约定 |
| DFT 与 FFT | $Y_k=\sum_n y_ne^{-2\pi ink/N}$；$y_n=N^{-1}\sum_kY_ke^{+2\pi ink/N}$ | 与 NumPy 默认约定一致；`fftshift` 仅重排索引，不改变变换定义 |
| 数学互相关 | $(f\star g)(\tau)=\int f^*(t)g(t+\tau)dt$ | 工程相关积 $V_{pq}=\langle E_pE_q^*\rangle$ 的共轭次序对应 $(E_q\star E_p)(0)$；积分与时间平均的归一化需另行说明，不能仅依据下标判断相位符号 |
| 复统计 | $\boldsymbol C=\mathbb E[(\boldsymbol z-\boldsymbol\mu)(\boldsymbol z-\boldsymbol\mu)^H]$，$\boldsymbol P=\mathbb E[(\boldsymbol z-\boldsymbol\mu)(\boldsymbol z-\boldsymbol\mu)^T]$ | 伪协方差为零的复高斯噪声通常称为 proper 复高斯噪声，此时 $\boldsymbol P=0$；标量 $\sigma_c^2=\mathbb E|z-\mu|^2$，其实部和虚部方差各为 $\sigma_c^2/2$ |
| 统计权重 | $\boldsymbol r=\boldsymbol d-\boldsymbol m$，$\chi^2=\boldsymbol r^H\boldsymbol C^{-1}\boldsymbol r$，$\boldsymbol W_{\rm stat}=\boldsymbol C^{-1}$ | 独立复样本有 $w_i=1/\sigma_{c,i}^2$；被标记剔除（flag）的样本应从似然中排除，在对角权重表示中等价于令 $w_i=0$ |
| 成像重加权 | $w_i^{\rm image}=q_iw_i^{\rm stat}$，其中 $q_i$ 可表示采样密度、Briggs 稳健加权或渐缩加权（taper）因子 | 重加权会改变点扩散函数（PSF）、噪声与尺度响应，不能再解释为原始噪声协方差的逆 |
| 向量与伴随 | 向量默认为列向量，$T$ 为转置，$*$ 为标量共轭，$H$ 为共轭转置 | 对复测量方程，正规方程和协方差必须使用 $H$ |

基线方向、可见度下标、斯托克斯参数 $V$ 和亮度矩阵归一化还涉及仪器定义，将在第 4 与第 7 章的章首约定表中统一说明；这些定义必须与本节的傅里叶变换和相关约定显式相容。

#### 本章结构

第 2.1 至 2.2 节建立基本表示语言。复数、相量和常见函数原型用于描述平面波、相位差、点源、主波束、有限孔径和采样过程，是后续技术章节的共同基础。

第 2.3 至 2.7 节构成傅里叶分析主线。傅里叶级数从周期信号出发，连续傅里叶变换将其推广到非周期函数，卷积、相关和傅里叶定理进一步统一描述信号与仪器响应之间的关系。这些内容为理解可见度空间、脏图像、脏波束和孔径合成提供数学基础。

第 2.8 至 2.9 节讨论离散化与数值实现。真实观测只提供有限、离散且带噪的样本，因此 DFT、FFT、采样定理和混叠直接限定成像能力与数值实现。谱泄漏、周期延拓、网格化误差和边界效应均可由这些离散模型解释。

第 2.10、2.11 和 2.15 节将前述函数关系组织为矩阵、优化与统计问题，为后续测量方程、正规方程、协方差、参数估计和迭代求解奠定基础。射电干涉测量通常需要在采样不完整、噪声显著和参数尺度差异较大的条件下获得稳定解。

第 2.12 至 2.13 节为几何补充专题，分别讨论辐射测量中的立体角和第 3 章所需的天球几何。第 2.14 节通过一维 CLEAN 计算实验联系傅里叶变换、采样、正规方程和病态反演。章末包括综合问题集、分段卷积示例和 FFT 编程作业。

#### 章节导航

1. [2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)
2. [2.2 干涉测量中常见的函数原型](2_2_important_functions.ipynb)
3. [2.3 傅里叶级数：从周期展开到谱表示](2_3_fourier_series.ipynb)
4. [2.4 连续傅里叶变换：从信号表示到可见度语言](2_4_the_fourier_transform.ipynb)
5. [2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)
6. [2.6 互相关、自相关与相似性度量](2_6_cross_correlation_and_auto_correlation.ipynb)
7. [2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
8. [2.8 离散傅里叶变换与 FFT：从解析公式到可计算算法](2_8_the_discrete_fourier_transform.ipynb)
    - [编程作业：实现基 2 FFT](fft_implementation_assignment.ipynb)
9. [2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)
10. [2.10 线性代数：从测量方程到矩阵表示](2_10_linear_algebra.ipynb)
11. [2.11 最小二乘与参数估计](2_11_least_squares.ipynb)
12. [2.12 补充专题：立体角与天球面积元素](2_12_solid_angle.ipynb)
13. [2.13 补充专题：球面三角学](2_13_spherical_trigonometry.ipynb)
14. [2.14 补充专题：一维 CLEAN 的数学演示](2_14_CLEAN_in_1D.ipynb)
15. [2.15 统计、不确定度、正则化与过拟合](2_15_statistics_uncertainty_regularization.ipynb)
16. [2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)
17. [2.y 综合练习与示例](2_y_exercises.ipynb)


***

下一节：[2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)
